
# NovaCart: Does a Smarter Shopping Page Convert Better?

### A/B Testing and Statistical Analysis

**Objective:** Evaluate whether a redesigned e-commerce experience improves purchase conversion.

**Tools:** Python, Pandas, NumPy, SciPy, Statsmodels, Matplotlib, Seaborn.

**Data transparency:** This project uses synthetic data generated for educational purposes. Results demonstrate statistical methods and are not evidence of actual customer behavior.

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep

from IPython.display import display

# Reproducible results
np.random.seed(42)

# Chart styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

print("NovaCart A/B testing environment is ready!")

NovaCart A/B testing environment is ready!


In [2]:

# Number of simulated visitors
N_VISITORS = 15_000

# Generate unique visitor IDs
visitor_ids = np.arange(1, N_VISITORS + 1)

# Randomly assign visitors to the two experiment groups
variants = np.random.choice(
    ["Control", "Treatment"],
    size=N_VISITORS,
    p=[0.5, 0.5]
)

# Visitor characteristics
devices = np.random.choice(
    ["Mobile", "Desktop", "Tablet"],
    size=N_VISITORS,
    p=[0.62, 0.32, 0.06]
)

traffic_sources = np.random.choice(
    ["Organic Search", "Social Media", "Paid Ads", "Direct"],
    size=N_VISITORS,
    p=[0.35, 0.25, 0.20, 0.20]
)

countries = np.random.choice(
    ["India", "USA", "UK", "Canada", "Australia"],
    size=N_VISITORS,
    p=[0.50, 0.20, 0.12, 0.10, 0.08]
)

# Base conversion probability for each visitor
conversion_probability = np.full(N_VISITORS, 0.075)

# Simulated differences by device
conversion_probability += np.where(devices == "Desktop", 0.025, 0)
conversion_probability += np.where(devices == "Tablet", 0.010, 0)

# Simulated differences by traffic source
conversion_probability += np.where(
    traffic_sources == "Direct", 0.025, 0
)
conversion_probability += np.where(
    traffic_sources == "Paid Ads", -0.010, 0
)

# Set the assumed treatment effect: +1.5 percentage points
conversion_probability += np.where(
    variants == "Treatment", 0.015, 0
)

# Keep probabilities within valid bounds
conversion_probability = np.clip(
    conversion_probability, 0.01, 0.50
)

# Simulate whether each visitor makes a purchase
converted = np.random.binomial(
    n=1,
    p=conversion_probability
)

# Simulate session duration in seconds
session_duration = np.random.gamma(
    shape=2.5,
    scale=80,
    size=N_VISITORS
).astype(int)

# Simulate pages viewed per session
pages_viewed = np.random.poisson(
    lam=3.2,
    size=N_VISITORS
) + 1

# Generate order values only for completed purchases
order_value = np.where(
    converted == 1,
    np.random.gamma(shape=4.0, scale=500, size=N_VISITORS).round(2),
    0.0
)

# Assemble the dataset
df = pd.DataFrame({
    "visitor_id": visitor_ids,
    "variant": variants,
    "device": devices,
    "traffic_source": traffic_sources,
    "country": countries,
    "converted": converted,
    "session_duration_seconds": session_duration,
    "pages_viewed": pages_viewed,
    "order_value": order_value
})

print(f"Generated {len(df):,} synthetic shopping sessions.")
print("Dataset shape:", df.shape)

display(df.head(10))

Generated 15,000 synthetic shopping sessions.
Dataset shape: (15000, 9)


,visitor_id,variant,device,traffic_source,country,converted,session_duration_seconds,pages_viewed,order_value
0,1,Control,Mobile,Paid Ads,India,0,499,3,0.0
1,2,Treatment,Desktop,Social Media,UK,0,215,4,0.0
2,3,Treatment,Mobile,Direct,Australia,0,65,2,0.0
3,4,Treatment,Mobile,Organic Search,Canada,0,148,3,0.0
4,5,Control,Mobile,Social Media,India,0,203,6,0.0
5,6,Control,Mobile,Paid Ads,UK,0,61,4,0.0
6,7,Control,Mobile,Direct,USA,0,203,2,0.0
7,8,Treatment,Mobile,Social Media,UK,0,61,3,0.0
8,9,Treatment,Mobile,Paid Ads,India,0,209,2,0.0
9,10,Treatment,Mobile,Organic Search,USA,0,153,7,0.0


In [3]:

print("Dataset dimensions:", df.shape)

print("\nData types and non-null counts:")
df.info()

print("\nMissing values per column:")
display(df.isnull().sum().to_frame("Missing values"))

print("\nDuplicate visitor IDs:", df["visitor_id"].duplicated().sum())

print("\nExperiment group sizes:")
display(df["variant"].value_counts().to_frame("Visitors"))

print("\nConversion values:")
print(sorted(df["converted"].unique()))

print("\nPurchase values for non-buyers:")
print(
    "All non-buyers have zero order value:",
    (df.loc[df["converted"] == 0, "order_value"] == 0).all()
)

Dataset dimensions: (15000, 9)

Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   visitor_id                15000 non-null  int64  
 1   variant                   15000 non-null  object 
 2   device                    15000 non-null  object 
 3   traffic_source            15000 non-null  object 
 4   country                   15000 non-null  object 
 5   converted                 15000 non-null  int64  
 6   session_duration_seconds  15000 non-null  int64  
 7   pages_viewed              15000 non-null  int64  
 8   order_value               15000 non-null  float64
dtypes: float64(1), int64(4), object(4)
memory usage: 1.0+ MB

Missing values per column:


,Missing values
visitor_id,0
variant,0
device,0
traffic_source,0
country,0
converted,0
session_duration_seconds,0
pages_viewed,0
order_value,0



Duplicate visitor IDs: 0

Experiment group sizes:


,Visitors
variant,
Control,7587
Treatment,7413



Conversion values:
[np.int64(0), np.int64(1)]

Purchase values for non-buyers:
All non-buyers have zero order value: True


In [4]:

summary = (
    df.groupby("variant")
    .agg(
        visitors=("visitor_id", "count"),
        conversions=("converted", "sum"),
        total_revenue=("order_value", "sum")
    )
)

summary["conversion_rate"] = (
    summary["conversions"] / summary["visitors"]
)

summary["conversion_rate_percent"] = (
    summary["conversion_rate"] * 100
)

summary["revenue_per_visitor"] = (
    summary["total_revenue"] / summary["visitors"]
)

display(summary.round(4))

,visitors,conversions,total_revenue,conversion_rate,conversion_rate_percent,revenue_per_visitor
variant,,,,,,
Control,7587,639,1235970.54,0.0842,8.4223,162.9064
Treatment,7413,712,1443170.72,0.0960,9.6047,194.6811
